# Week 3 — Network Data & Spatial Data

**Course:** Data Visualization
**Topics:** Network/graph visualization (node-link diagrams, force-directed layout,
adjacency matrices) and spatial data visualization (choropleth maps, proportional
symbol maps, dot density maps) — based on Munzner's *Visualization Analysis &
Design*, Ch. 9.

---

## Overview

This lab has two independent parts that mirror the two halves of this week's
lecture slides:

1. **Network data** — we treat Ecuador's domestic flight network as a graph:
   airports are **nodes**, routes between them are **links**. We build a
   node-link diagram, apply a force-directed layout, and compare it against an
   adjacency matrix — the same network, two different idioms, each with its own
   strengths (per the lecture's comparison of node-link vs. matrix
   representations).

2. **Spatial data** — we work with two real Ecuadorian government datasets at
   the **canton** level (Ecuador's 224 third-level administrative divisions),
   joined against an official canton-boundary GeoJSON:
   - **MAATE** (Ministry of Environment) conservation-area data → a
     **choropleth map**
   - **MDI** (Ministry of the Interior) detentions/apprehensions data, which
     includes exact latitude/longitude per incident → a **proportional symbol
     map** and a **dot density map**, plus a direct demonstration of why
     normalizing by population matters (the lecture's central warning about
     choropleth maps).

All datasets are downloaded directly inside the notebook from a public GitHub
repository, so the notebook runs top-to-bottom on a fresh Colab instance with no
manual file uploads.

**Grading:** Each activity is worth the same number of points.


## 1. Setup

We use **Plotly** for every chart in this notebook (ships pre-installed on
Colab) and **NetworkX** for the force-directed graph layout algorithm used in
Part 1 — NetworkX is *not* always part of the default Colab image, so we
install it defensively (a no-op if it's already present).


In [ ]:
# NetworkX computes the force-directed (spring) layout for the airport network
# in Part 1. Installed defensively since it isn't always preloaded on Colab.
try:
    import networkx as nx
except ImportError:
    !pip install networkx
    import networkx as nx


In [ ]:
import json
import unicodedata
import re
import urllib.request
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os, sys
import plotly.io as pio

# ── Plotly renderer auto-detection ──────────────────────────────────────────
# VS Code + Colab plugin: JPY_SESSION_NAME is a /content/ file path.
# Colab browser: JPY_SESSION_NAME is a bare Google Drive file ID (no /content/ prefix).
_jpy_session = os.environ.get('JPY_SESSION_NAME', '')
if _jpy_session.startswith('/content/'):
    pio.renderers.default = 'vscode'   # VS Code + Colab plugin
elif 'google.colab' in sys.modules:
    pio.renderers.default = 'colab'    # Colab in browser
else:
    pio.renderers.default = 'notebook' # local Jupyter fallback

print(f"Using Plotly renderer: {pio.renderers.default}")

---

# Part 1 — Network Data

## 2. Loading Airports and Routes

We use the **OpenFlights** dataset — a well-known open dataset of the world's
airports and airline routes, widely used as a teaching example for network
visualization because the nodes (airports) come with real geographic
coordinates, which lets us sanity-check any graph layout against the real map.

Two files, hosted on GitHub so the notebook needs no manual upload:

- **`airports.dat.txt`** — one row per airport worldwide: ID, name, city,
  country, IATA/ICAO codes, latitude, longitude, and more.
- **`routes.dat.txt`** — one row per airline route: origin airport, destination
  airport, airline, and equipment type. Neither file ships with a header row
  (a quirk of the original OpenFlights format), so we supply column names
  ourselves.


In [ ]:
AIRPORTS_URL = ('https://raw.githubusercontent.com/CarlaGeovanna/'
                 'datasets_public/main/airports.dat.txt')
ROUTES_URL = ('https://raw.githubusercontent.com/CarlaGeovanna/'
              'datasets_public/refs/heads/main/routes.dat.txt')

airport_cols = ['airport_id', 'name', 'city', 'country', 'iata', 'icao',
                'lat', 'lon', 'altitude', 'timezone', 'dst', 'tz_name',
                'type', 'source']
airports = pd.read_csv(AIRPORTS_URL, header=None, names=airport_cols, na_values='\\N')

route_cols = ['airline', 'airline_id', 'src_iata', 'src_id', 'dst_iata',
              'dst_id', 'codeshare', 'stops', 'equipment']
routes = pd.read_csv(ROUTES_URL, header=None, names=route_cols, na_values='\\N')

print(f"Airports worldwide: {len(airports):,}")
print(f"Routes worldwide: {len(routes):,}")
airports.head()


## 3. Filtering to Ecuador's Domestic Network

The full worldwide graph has ~7,700 airports and ~67,600 routes — far too dense
to read as a node-link diagram. Per this week's lecture, force-directed layouts
only stay legible for **small, sparse graphs** (the slides cite roughly
`E < 4N`, edges under four times the node count), so we scope the network down
to something a node-link diagram can actually show clearly: **Ecuador's
domestic flight network** — airports located in Ecuador, and only the routes
that connect two of them.

This keeps every node's position geographically meaningful (we can later
compare the force-directed layout against each airport's true lat/lon) and
keeps the graph small enough that edge crossings and clustering are easy to
read by eye. Proximity in space


In [ ]:
# Airports located in Ecuador
ec_airports = airports[airports['country'] == 'Ecuador'].copy()
ec_iata_codes = set(ec_airports['iata'].dropna())

# Routes where BOTH endpoints are Ecuadorian airports (domestic routes only)
ec_routes = routes[
    routes['src_iata'].isin(ec_iata_codes) & routes['dst_iata'].isin(ec_iata_codes)
].copy()

# Some airlines file duplicate route entries (different flight numbers, same
# city pair) — for a topology diagram we only care whether a connection
# EXISTS, not how many flights serve it, so we deduplicate to unique edges.
edges = ec_routes[['src_iata', 'dst_iata']].drop_duplicates().reset_index(drop=True)

# Only keep airports that actually appear in at least one domestic route —
# an airport with zero domestic connections would be an isolated, disconnected
# node that adds clutter without adding any topology to explore.
nodes_in_use = set(edges['src_iata']) | set(edges['dst_iata'])
ec_nodes = ec_airports[ec_airports['iata'].isin(nodes_in_use)].copy()

print(f"Ecuadorian airports with domestic connections: {len(ec_nodes)}")
print(f"Domestic routes (unique city pairs): {len(edges)}")
ec_nodes[['name', 'city', 'iata', 'lat', 'lon']]


## 4. Node-Link Diagram: Geographic Layout

Before trying a force-directed layout, we first draw the network using each
airport's **true geographic position** as the node-link layout. This is a
useful baseline: it tells us what the "ground truth" spatial structure looks
like, so we can later judge whether the force-directed layout preserves or
distorts it.

Per the lecture's node-link vocabulary: airports are **point marks** (nodes),
routes are **line marks** (links) drawn as connections between them.


In [ ]:
# Build a lookup so we can find each airport's lat/lon by IATA code
coord = ec_nodes.set_index('iata')[['lat', 'lon']]

# Build one continuous line trace for all edges, with None between segments —
# this is the standard Plotly trick for drawing many disconnected line
# segments in a single trace instead of one trace per edge (much faster to render).
edge_lat, edge_lon = [], []
for _, row in edges.iterrows():
    edge_lat += [coord.loc[row['src_iata'], 'lat'], coord.loc[row['dst_iata'], 'lat'], None]
    edge_lon += [coord.loc[row['src_iata'], 'lon'], coord.loc[row['dst_iata'], 'lon'], None]

fig = go.Figure()

# Links drawn first so nodes render on top of them
fig.add_trace(go.Scattergeo(
    lat=edge_lat, lon=edge_lon,
    mode='lines',
    line=dict(width=1, color='#898781'),
    hoverinfo='none',
    showlegend=False,
))

# Nodes as point marks, sized slightly larger for hub airports (more connections)
degree = pd.concat([edges['src_iata'], edges['dst_iata']]).value_counts()
ec_nodes['degree'] = ec_nodes['iata'].map(degree)

fig.add_trace(go.Scattergeo(
    lat=ec_nodes['lat'], lon=ec_nodes['lon'],
    mode='markers+text',
    text=ec_nodes['iata'],
    textposition='top center',
    marker=dict(size=ec_nodes['degree'] * 4 + 6, color='#2a78d6',
                line=dict(width=0.5, color='white')),
    hovertext=ec_nodes['name'] + ' (' + ec_nodes['city'] + ') — '
              + ec_nodes['degree'].astype(str) + ' connections',
    hoverinfo='text',
    showlegend=False,
))

fig.update_geos(
    lataxis_range=[-6, 2], lonaxis_range=[-92, -74],
    showcountries=True, countrycolor='#c3c2b7',
    showland=True, landcolor='#f9f9f7',
    showocean=True, oceancolor='#eaf2fb',
    coastlinecolor='#898781',
)
fig.update_layout(
    title="Ecuador's Domestic Flight Network — Geographic Layout<br>"
          "<sub>Node size = number of domestic connections (degree)</sub>",
    margin=dict(l=0, r=0, t=60, b=0), height=650,
)
fig.show()


### Reading the geographic layout

- **Quito (UIO)** and **Guayaquil (GYE)** are immediately visible as the two
  hub airports — the largest nodes, with the most connecting lines — matching
  their role as Ecuador's two largest cities and main air-traffic hubs.
- **Galápagos (GPS, SCY)** stands far apart from the mainland cluster, exactly
  where its true geography places it — this is the payoff of using real
  coordinates instead of an arbitrary layout: the map instantly explains why
  those two airports look "isolated" on the edge of the diagram.
- The geographic layout is easy to orient on (readers already know what
  Ecuador looks like), but as the lecture notes, geographic position is not
  always the layout that most clearly shows a network's *topology* — routes
  that are geographically short can still be topologically important, and
  vice versa.


## 5. Force-Directed (Spring) Layout

Now we lay out the *same* graph using a **force-directed placement**
algorithm — the idiom the lecture highlights as the most common approach for
general node-link diagrams. The physical intuition: think of every **link as a
spring** pulling connected airports together, and every **node as a magnet**
repelling all other nodes apart. The algorithm places nodes randomly, then
iterates until these opposing forces reach equilibrium.

Unlike the geographic layout, **spatial position here carries no direct
meaning** — the x/y coordinates are an artifact of the physics simulation, not
latitude/longitude. What the layout *is* good at, per the lecture, is
minimizing edge crossings and making clusters/hubs visually obvious — this is
the "explore topology, locate clusters" idiom.

We use NetworkX's `spring_layout`, which implements exactly this
algorithm (a Fruchterman-Reingold force-directed layout).


In [ ]:
# Build the graph structure: one node per Ecuadorian airport with a domestic
# connection, one edge per unique route (undirected — a route connects two
# airports regardless of flight direction).
G = nx.Graph()
G.add_nodes_from(ec_nodes['iata'])
G.add_edges_from(edges[['src_iata', 'dst_iata']].values)

# seed=42 makes the layout reproducible — force-directed placement is
# nondeterministic by nature (per the lecture's "weaknesses" slide), so
# without a fixed seed the exact node positions would differ on every re-run.
pos = nx.spring_layout(G, seed=42, k=0.6, iterations=200)

# Unpack into a DataFrame for easy plotting
layout_df = pd.DataFrame(pos).T.rename(columns={0: 'x', 1: 'y'})
layout_df['iata'] = layout_df.index
layout_df = layout_df.merge(ec_nodes[['iata', 'name', 'city', 'degree']], on='iata')

layout_df.head()


In [ ]:
fig_fd = go.Figure()

# Links as line segments between the force-directed (x, y) positions
fd_edge_x, fd_edge_y = [], []
for src, dst in edges[['src_iata', 'dst_iata']].values:
    fd_edge_x += [pos[src][0], pos[dst][0], None]
    fd_edge_y += [pos[src][1], pos[dst][1], None]

fig_fd.add_trace(go.Scatter(
    x=fd_edge_x, y=fd_edge_y,
    mode='lines', line=dict(width=1, color='#898781'),
    hoverinfo='none', showlegend=False,
))

fig_fd.add_trace(go.Scatter(
    x=layout_df['x'], y=layout_df['y'],
    mode='markers+text',
    text=layout_df['iata'], textposition='top center',
    marker=dict(size=layout_df['degree'] * 4 + 6, color='#2a78d6',
                line=dict(width=0.5, color='white')),
    hovertext=layout_df['name'] + ' (' + layout_df['city'] + ') — '
              + layout_df['degree'].astype(str) + ' connections',
    hoverinfo='text', showlegend=False,
))

fig_fd.update_xaxes(visible=False)
fig_fd.update_yaxes(visible=False)
fig_fd.update_layout(
    title="Ecuador's Domestic Flight Network — Force-Directed Layout<br>"
          "<sub>Position is NOT geographic — it minimizes edge crossings and reveals hub structure</sub>",
    plot_bgcolor='#fcfcfb', margin=dict(l=20, r=20, t=60, b=20), height=650,
)
fig_fd.show()


### Comparing the two layouts

Look at how **Quito (UIO)** and **Guayaquil (GYE)** are positioned in the
force-directed layout compared to the geographic one: the algorithm pulls hub
airports toward the center of the drawing — not because of where they sit on
the map, but because they have the *most springs* pulling on them from every
direction. Meanwhile a low-degree airport connected to just one hub gets
pushed to the outer edge of the layout, forming the same "spoke" pattern the
lecture's force-directed example shows.

This is the core trade-off the lecture calls out explicitly: the geographic
layout is easier to *orient* on (you already know the map), but the
force-directed layout more directly answers *topological* questions — "which
airports are the most connected?" is immediately visible as "which nodes sit
in the densest tangle," without needing to count lines by eye.


## 6. Detecting Communities

The lecture lists **"identify clusters / communities"** as one of the core
topology-based tasks for network data. So far we've relied on *eyeballing*
the force-directed layout to spot clusters — real analyses instead run a
**community-detection algorithm**, which partitions a graph's nodes into
groups that are more densely connected to each other than to the rest of the
network, without a human deciding the grouping by hand.

We use NetworkX's **Louvain algorithm** (`louvain_communities`), the standard
modern choice for this task — it repeatedly merges nodes into groups that
maximize *modularity*, a score that rewards partitions where most edges stay
inside a group rather than crossing between groups. We then recolor the same
force-directed layout from Section 5, one categorical hue per detected
community, following the pattern from the lecture's Les Misérables example
(`bost.ocks.org/mike/miserables`), where recoloring by detected community
turned an unreadable tangle into a diagram where the community structure
"just appears."


In [ ]:
from networkx.algorithms.community import louvain_communities

# seed=42 for reproducibility — like spring_layout, Louvain's merge order has
# a random component, so without a fixed seed the exact grouping could vary
# slightly (though the modularity score it converges to would not).
communities = louvain_communities(G, seed=42)

print(f"Communities detected: {len(communities)}")
for i, comm in enumerate(communities):
    print(f"  Community {i}: {sorted(comm)}")

# Map each airport to its community index, for use as a color key.
community_of = {node: i for i, comm in enumerate(communities) for node in comm}
layout_df['community'] = layout_df['iata'].map(community_of).astype(str)


In [ ]:
# First four slots of the course's validated categorical palette — this graph
# only has 3 communities, comfortably under the 4-category cap for CVD-safe
# categorical color.
community_colors = {'0': '#2a78d6', '1': '#008300', '2': '#e87ba4', '3': '#eda100'}

fig_comm = go.Figure()

# Links stay neutral gray — color is reserved for the community assignment,
# not the edges, so the community grouping is the one thing standing out.
fig_comm.add_trace(go.Scatter(
    x=fd_edge_x, y=fd_edge_y,
    mode='lines', line=dict(width=1, color='#c3c2b7'),
    hoverinfo='none', showlegend=False,
))

for comm_id in sorted(layout_df['community'].unique()):
    sub = layout_df[layout_df['community'] == comm_id]
    fig_comm.add_trace(go.Scatter(
        x=sub['x'], y=sub['y'],
        mode='markers+text', text=sub['iata'], textposition='top center',
        marker=dict(size=sub['degree'] * 4 + 6, color=community_colors[comm_id],
                    line=dict(width=0.5, color='white')),
        hovertext=sub['name'] + ' (' + sub['city'] + ')',
        hoverinfo='text', name=f'Community {comm_id}',
    ))

fig_comm.update_xaxes(visible=False)
fig_comm.update_yaxes(visible=False)
fig_comm.update_layout(
    title="Ecuador's Domestic Flight Network — Colored by Detected Community<br>"
          "<sub>Force-directed layout (same as Section 5), color = Louvain community assignment</sub>",
    plot_bgcolor='#fcfcfb', margin=dict(l=20, r=20, t=60, b=20), height=650,
)
fig_comm.show()


### Reading the communities

The algorithm finds three communities, and they line up neatly with what the
force-directed layout already hinted at visually: a **UIO-centered group**, a
**GYE-centered group**, and a small **LTX–OCC pair** that connects only to
each other and nothing else in this domestic subset — modularity correctly
isolates it as its own community rather than forcing it into one of the two
big hubs.

This is the payoff of running a real algorithm instead of eyeballing the
layout: with only 14 nodes we could probably have guessed this grouping by
eye, but the same Louvain call scales to networks with thousands of nodes,
where visual clustering stops being reliable — exactly the scalability
argument the lecture makes for algorithmic community detection over manual
inspection.


### Does edge weight change the answer?

So far every edge in `G` counts equally — a route flown by one small
regional carrier weighs the same as Quito–Guayaquil, served by many
overlapping airlines. But `routes.dat` actually encodes a real signal of
connection *strength*: how many separate airline route entries exist for
each city pair. We didn't use this signal yet — the deduplication step in
Section 3 collapsed it away on purpose, to keep the first pass at the graph
simple. Now we bring it back as an **edge weight**, and ask whether it
changes which communities the algorithm finds.


In [ ]:
# Recompute the per-city-pair route-entry count directly from ec_routes
# (the un-deduplicated table from Section 3) — how many separate airline
# entries connect each pair, regardless of flight direction.
pair_counts = (
    ec_routes.assign(
        pair=ec_routes.apply(lambda r: tuple(sorted([r['src_iata'], r['dst_iata']])), axis=1)
    )
    .groupby('pair')
    .size()
    .reset_index(name='n_route_entries')
)
pair_counts.sort_values('n_route_entries', ascending=False)


In [ ]:
# Rebuild the graph with weighted edges — same topology as G, but each edge
# now carries its route-entry count as a 'weight' attribute.
G_weighted = nx.Graph()
for _, row in pair_counts.iterrows():
    src, dst = row['pair']
    G_weighted.add_edge(src, dst, weight=row['n_route_entries'])

# Louvain with weight=None reproduces the unweighted result from above;
# weight='weight' lets high-traffic edges pull harder during merging.
communities_weighted = louvain_communities(G_weighted, weight='weight', seed=42)

print(f"Communities detected (weighted): {len(communities_weighted)}")
for i, comm in enumerate(communities_weighted):
    print(f"  Community {i}: {sorted(comm)}")


To make the weight itself visible — not just its effect on the community
split — we draw each edge as its **own** line segment (rather than one
combined trace, as in the earlier figures) and scale its thickness linearly
with `n_route_entries`. We also add real hover text to both nodes and edges,
so the exact numbers behind the picture are one hover away instead of living
only in the `pair_counts` table above.


In [ ]:
# Recolor the SAME force-directed layout (pos, from Section 5) using the
# weighted community assignment — the node positions don't change, only the
# color grouping does, which makes the two community structures directly
# comparable side by side against the earlier unweighted figure.
community_of_weighted = {node: i for i, comm in enumerate(communities_weighted) for node in comm}
layout_df['community_weighted'] = layout_df['iata'].map(community_of_weighted).astype(str)

fig_comm_w = go.Figure()

# One trace PER EDGE (not a single combined trace) — this is what lets each
# line have its own width and its own hover text. A simple linear scale
# (weight * constant) maps 'n_route_entries' to line thickness: GYE-UIO's
# 13 entries draw an 8.5px line, a single-entry route draws a 1.5px line.
for _, row in pair_counts.iterrows():
    src, dst = row['pair']
    weight = row['n_route_entries']
    line_width = 1 + weight * 0.6
    fig_comm_w.add_trace(go.Scatter(
        x=[pos[src][0], pos[dst][0]], y=[pos[src][1], pos[dst][1]],
        mode='lines',
        line=dict(width=line_width, color='#c3c2b7'),
        hovertext=f"{src}–{dst}: {weight} route entr{'y' if weight == 1 else 'ies'}",
        hoverinfo='text', showlegend=False,
    ))

for comm_id in sorted(layout_df['community_weighted'].unique()):
    sub = layout_df[layout_df['community_weighted'] == comm_id]
    fig_comm_w.add_trace(go.Scatter(
        x=sub['x'], y=sub['y'],
        mode='markers+text', text=sub['iata'], textposition='top center',
        marker=dict(size=sub['degree'] * 4 + 6, color=community_colors[comm_id],
                    line=dict(width=0.5, color='white')),
        hovertext=(sub['name'] + ' (' + sub['city'] + ')<br>'
                   + sub['degree'].astype(str) + ' domestic connections<br>'
                   + 'Community ' + sub['community_weighted']),
        hoverinfo='text', name=f'Community {comm_id}',
    ))

fig_comm_w.update_xaxes(visible=False)
fig_comm_w.update_yaxes(visible=False)
fig_comm_w.update_layout(
    title="Ecuador's Domestic Flight Network — Colored by WEIGHTED Community<br>"
          "<sub>Same layout as above · line thickness = route-entry count (edge weight) · "
          "hover for details</sub>",
    plot_bgcolor='#fcfcfb', margin=dict(l=20, r=20, t=60, b=20), height=650,
)
fig_comm_w.show()


Weighting the edges collapses the network from **three** communities down to
**two**: `CUE` and `ESM`, which the unweighted version had grouped with GYE,
switch sides and join the UIO group instead. Looking back at
`pair_counts`, the reason is visible directly in the numbers — CUE–UIO and
ESM–UIO each have as many or more route entries as CUE–GYE and ESM–GYE, so
once that traffic volume counts toward the merge decision, those two airports
pull harder toward Quito's side of the network than the topology alone
suggested.

This is a genuinely useful check to run on any network: **an unweighted
community structure only reflects which connections exist, not how strong
they are** — and for a real-world network like a flight route map, the two
can disagree. Whenever the dataset actually contains a meaningful weight
(traffic volume, frequency, capacity), it's worth re-running community
detection with it before trusting the unweighted partition.


## 7. Adjacency Matrix Representation

The third idiom from the lecture: instead of drawing nodes and links, we
**derive a table** from the network — one row and one column per airport,
where cell `(i, j)` is shaded if a route connects airport `i` to airport `j`.




In [ ]:
# Build the adjacency matrix directly from the graph — order matters for
# readability (the lecture's "Node order is crucial" slide), so we order
# airports by degree (most-connected first) rather than alphabetically,
# which visually groups the hub structure near the matrix's top-left corner.
node_order = layout_df.sort_values('degree', ascending=False)['iata'].tolist()
adj = nx.to_pandas_adjacency(G, nodelist=node_order)

fig_matrix = px.imshow(
    adj,
    color_continuous_scale=[[0, '#fcfcfb'], [1, '#2a78d6']],
    aspect='equal',
)
fig_matrix.update_layout(
    title="Ecuador's Domestic Flight Network — Adjacency Matrix<br>"
          "<sub>Ordered by degree (most-connected airports first) — same network as above</sub>",
    coloraxis_showscale=False,
    xaxis_title='', yaxis_title='',
    height=600, width=650,
)
fig_matrix.show()


### Reading the matrix

Because we ordered airports by degree, the hub airports' rows/columns
(**UIO**, **GYE**) show up as nearly-full rows and columns near the top-left —
you can immediately see they connect to almost everything, without needing to
trace a single line. Try finding **UIO's neighbors** in the matrix (scan its
row) versus finding them in the node-link diagrams above — the matrix answers
that question faster. Now try tracing a **two-hop path** from one low-degree
airport to another that isn't a direct route — this is where the matrix
becomes noticeably harder to use than the node-link diagram, exactly as the
lecture predicts.


### Where the Matrix Actually Wins: A Denser Network

With only 14 airports, the node-link diagram was never really in trouble —
both idioms stayed readable, so the comparison above couldn't show the
matrix's real advantage. The lecture is specific about when that advantage
kicks in: node-link diagrams work well for **small, sparse graphs**, but
matrices become the better choice once a network gets **too dense to draw
without becoming visual noise** (the lecture's empirical study cites node-link
as best for small networks, matrix as best for large ones — provided the task
doesn't require path-tracing).

To see that trade-off honestly, we scale up: instead of Ecuador alone, we
build the domestic-and-regional flight network for **all of South America** —
still the same `airports` / `routes` tables from Section 2, just without the
`country == 'Ecuador'` filter from Section 3.


In [ ]:
# All South American airports (not just Ecuador), and all routes where BOTH
# endpoints are in South America — this reuses the exact same 'airports' and
# 'routes' tables loaded once in Section 2, just with a wider country filter.
south_america = [
    'Ecuador', 'Colombia', 'Peru', 'Venezuela', 'Brazil', 'Bolivia', 'Chile',
    'Argentina', 'Uruguay', 'Paraguay', 'Guyana', 'Suriname', 'French Guiana',
]
sa_airports = airports[airports['country'].isin(south_america)].copy()
sa_iata_codes = set(sa_airports['iata'].dropna())

sa_routes = routes[
    routes['src_iata'].isin(sa_iata_codes) & routes['dst_iata'].isin(sa_iata_codes)
].copy()
sa_edges = sa_routes[['src_iata', 'dst_iata']].drop_duplicates().reset_index(drop=True)

sa_nodes_in_use = set(sa_edges['src_iata']) | set(sa_edges['dst_iata'])
sa_nodes = sa_airports[sa_airports['iata'].isin(sa_nodes_in_use)].copy()

print(f"South American airports with connections: {len(sa_nodes)}")
print(f"South American unique routes: {len(sa_edges)}")


First, the node-link diagram — same geographic-layout approach as Section 4,
just at 20x the scale.


In [ ]:
sa_coord = sa_nodes.set_index('iata')[['lat', 'lon']]

sa_edge_lat, sa_edge_lon = [], []
for _, row in sa_edges.iterrows():
    sa_edge_lat += [sa_coord.loc[row['src_iata'], 'lat'], sa_coord.loc[row['dst_iata'], 'lat'], None]
    sa_edge_lon += [sa_coord.loc[row['src_iata'], 'lon'], sa_coord.loc[row['dst_iata'], 'lon'], None]

fig_sa = go.Figure()
fig_sa.add_trace(go.Scattergeo(
    lat=sa_edge_lat, lon=sa_edge_lon, mode='lines',
    line=dict(width=0.4, color='#c3c2b7'), opacity=0.5,
    hoverinfo='none', showlegend=False,
))
fig_sa.add_trace(go.Scattergeo(
    lat=sa_nodes['lat'], lon=sa_nodes['lon'],
    mode='markers', marker=dict(size=3, color='#2a78d6'),
    hovertext=sa_nodes['name'] + ' (' + sa_nodes['city'] + ')',
    hoverinfo='text', showlegend=False,
))
fig_sa.update_geos(
    scope='south america',
    showcountries=True, countrycolor='#c3c2b7',
    showland=True, landcolor='#f9f9f7',
    showocean=True, oceancolor='#eaf2fb', coastlinecolor='#898781',
)
fig_sa.update_layout(
    title=f"South America's Flight Network — Node-Link Diagram<br>"
          f"<sub>{len(sa_nodes)} airports, {len(sa_edges)} routes — compare readability to the 14-node Ecuador map above</sub>",
    margin=dict(l=0, r=0, t=60, b=0), height=700,
)
fig_sa.show()


At this density, the node-link diagram becomes exactly the "visual noise"
the lecture warns about: hundreds of overlapping line segments make it
essentially impossible to trace any individual airport's connections, and
even the largest hubs are hard to distinguish from the general clutter.
Zooming in would help locally, but there is no single static layout at this
scale that stays readable everywhere at once.

Now the same network as an adjacency matrix.


In [ ]:
G_sa = nx.Graph()
G_sa.add_nodes_from(sa_nodes['iata'])
G_sa.add_edges_from(sa_edges[['src_iata', 'dst_iata']].values)

# Same degree-first ordering strategy as the Ecuador-only matrix — critical
# here, since with 300+ nodes an arbitrary (e.g. alphabetical) order would
# scatter the hub structure randomly across the matrix instead of grouping it.
sa_degree = dict(G_sa.degree())
sa_node_order = sorted(G_sa.nodes(), key=lambda n: -sa_degree[n])
adj_sa = nx.to_pandas_adjacency(G_sa, nodelist=sa_node_order)

fig_matrix_sa = px.imshow(
    adj_sa,
    color_continuous_scale=[[0, '#fcfcfb'], [1, '#2a78d6']],
    aspect='equal',
)
fig_matrix_sa.update_layout(
    title="South America's Flight Network — Adjacency Matrix<br>"
          "<sub>Ordered by degree (most-connected airports first) — same network as the map above</sub>",
    coloraxis_showscale=False,
    xaxis=dict(showticklabels=False, title=''),
    yaxis=dict(showticklabels=False, title=''),
    height=650, width=650,
)
fig_matrix_sa.show()


### Reading the South America matrix

With node labels hidden (300+ airports won't fit as readable tick labels
anyway — a scalability limit of its own), the structure that matters is still
immediately visible: a **dense block in the top-left corner**, where the
handful of major hubs (São Paulo/GRU, Bogotá/BOG, Buenos Aires/AEP, Lima/LIM —
the same cities that topped the degree ranking earlier) connect to nearly
everything, fading into a **sparse, scattered field** toward the bottom-right,
where most airports connect to only one or two hubs and nothing else.

This is precisely the pattern the node-link diagram *couldn't* show cleanly at
this scale — the matrix compresses hundreds of edges into a single glance-able
image, at the cost of exactly the weakness the lecture predicts: you can no
longer easily answer "how do I fly from a small Bolivian airstrip to a small
Paraguayan one?" by tracing a path — that question needs the node-link
diagram, or a routing algorithm, not the matrix. Neither idiom is
strictly better; each earns its place for a different task, which is the
lecture's core point about node-link vs. matrix representations.


### Coloring the Matrix by Community

Degree-ordering already revealed the hub structure, but it says nothing about
**which airports cluster together** — two airports can both be high-degree
without belonging to the same regional sub-network. Section 6 already showed
how to detect that with Louvain community detection; the lecture's own
"Node order is crucial: Reordering" slide (the *Les Misérables* example) makes
the same point for matrices specifically: **reordering rows/columns by
detected community**, instead of by degree, turns the block-diagonal
structure into something you can literally see — each community becomes a
solid square sitting on the diagonal.


In [ ]:
# Detect communities on the South America graph — same algorithm as Section 6,
# just applied to a much larger network.
sa_communities = louvain_communities(G_sa, seed=42)
sa_communities = sorted(sa_communities, key=len, reverse=True)  # largest first

print(f"Communities detected: {len(sa_communities)}")
for i, comm in enumerate(sa_communities):
    comm_countries = sa_nodes[sa_nodes['iata'].isin(comm)]['country'].value_counts()
    print(f"  Community {i} (size {len(comm)}): {dict(comm_countries)}")


Nine communities emerge, and — unlike the small Ecuador example, where
communities were really just "which hub do you connect to" — at this scale
the communities line up almost exactly with **countries and regional blocks**:
Colombia and Ecuador merge into one community (the Bogotá–Quito corridor is
evidently strong enough to outweigh national borders), Chile and Bolivia form
another, and Argentina groups with Paraguay and Uruguay. Brazil — the largest
country in the dataset — splits into three separate communities, hinting at
internal regional structure (likely a north/northeast cluster, a
São-Paulo-centered southeast cluster, and a smaller peripheral one) rather
than one single national network.


In [ ]:
# Reorder the matrix by community FIRST, then by degree within each community —
# this groups every community into a contiguous, near-diagonal block, and
# orders each block internally so its own local hub still sits at the top-left.
community_of_sa = {node: i for i, comm in enumerate(sa_communities) for node in comm}
community_order = sorted(
    G_sa.nodes(),
    key=lambda n: (community_of_sa[n], -sa_degree[n])
)
adj_sa_comm = nx.to_pandas_adjacency(G_sa, nodelist=community_order)

# A qualitative palette with enough distinct hues for 9 communities — at this
# node count the color's job is to make block boundaries visible, not to pass
# a strict CVD-pairwise check the way a 3-4 category chart would.
community_palette = px.colors.qualitative.Set1 + px.colors.qualitative.Set3
community_ids = [community_of_sa[n] for n in community_order]

# Recolor each cell: 0 stays "no connection" (background), 1 becomes the
# ROW airport's community index + 1 — so filled cells inherit their row's
# community color instead of a single flat blue.
adj_colored = adj_sa_comm.copy()
for i, node in enumerate(community_order):
    adj_colored.iloc[i] = adj_colored.iloc[i] * (community_of_sa[node] + 1)

fig_matrix_comm = px.imshow(
    adj_colored,
    color_continuous_scale=[[0, '#fcfcfb']] + [
        [(_i + 1) / len(sa_communities), community_palette[_i]]
        for _i in range(len(sa_communities))
    ],
    aspect='equal',
)
fig_matrix_comm.update_layout(
    title="South America's Flight Network — Adjacency Matrix Colored by Community<br>"
          "<sub>Rows/columns grouped by Louvain community, then by degree within each — "
          "compare block boundaries to the country breakdown above</sub>",
    coloraxis_showscale=False,
    xaxis=dict(showticklabels=False, title=''),
    yaxis=dict(showticklabels=False, title=''),
    height=650, width=650,
)
fig_matrix_comm.show()


In [ ]:
# imprime la data de aeropuertos en sudamerica
print(sa_nodes)

# expresa quien es BSB y VCP de sa_nodes que paises son
print(sa_nodes[sa_nodes['iata'].isin(['BSB', 'VCP'])]['country'].value_counts())


### Reading the community-colored matrix

Reordering by community turns the diffuse pattern from the degree-only matrix
into **visible blocks along the diagonal** — each solid-colored square is one
detected community, and because most of a community's connections stay
*within* the community, the vast majority of colored cells cluster inside
these blocks rather than scattering across the whole matrix. The faint marks
*outside* the diagonal blocks are the cross-community routes — the direct
flights that cross a "national" boundary in the network's structure, like a
Bogotá–Lima or Buenos Aires–São Paulo connection. Those off-diagonal marks are
comparatively rare, which is exactly what "community" means in graph terms:
dense connections inside the group, sparse connections crossing between
groups.


---

# Part 2 — Spatial Data

## 8. Loading Canton Boundaries

Both spatial datasets in this part are joined against the same official
**canton-level GeoJSON** (downloaded from the course dataset repository,
same as every other dataset in this notebook) — 224 polygons, one per
Ecuadorian canton, each carrying a `DPA_CANTON` code (e.g. `"0101"`) that we
will use as the join key. Loading it once here keeps Section 9 (choropleth)
and Section 10 (symbol map / dot density) from duplicating this step.

In [ ]:
CANTONES_GEOJSON_URL = ('https://raw.githubusercontent.com/CarlaGeovanna/'
                         'datasets_public/refs/heads/main/cantones-ecuador.geojson')

with urllib.request.urlopen(CANTONES_GEOJSON_URL) as response:
    cantones_geojson = json.load(response)

print(f"Canton polygons loaded: {len(cantones_geojson['features'])}")
print("Sample properties:", cantones_geojson['features'][0]['properties'])

## 9. Choropleth Map: Conservation Area by Canton (MAATE)

Our first spatial dataset comes from Ecuador's **Ministry of Environment,
Water and Ecological Transition (MAATE)**: the surface area (in hectares)
under some form of conservation status, recorded per canton/parish as of
December 2022. Conservation categories include `SNAP` (National System of
Protected Areas), `PSB` (Socio Bosque forest-conservation program), `MANGLAR`
(mangrove), and `BVP` (Protective Forest and Vegetation).

This is a natural fit for a **choropleth map** — per the lecture, choropleths
work best when the central task is understanding *spatial relationships* in a
single quantitative variable per region, and conservation area (a genuine
per-region total, not a population-driven count) is exactly that kind of
variable.


In [ ]:
MAATE_URL = ('https://raw.githubusercontent.com/CarlaGeovanna/'
             'datasets_public/main/maate_superficiebajoconservacion_2022diciembre.csv')

maate = pd.read_csv(MAATE_URL, sep=';', encoding='latin-1')

# AREA_HA is stored with a comma as the decimal separator (a common convention
# in Spanish-language government exports) — pandas would otherwise read this
# column as text instead of a number.
maate['AREA_HA'] = maate['AREA_HA'].str.replace(',', '.').astype(float)

print(f"Rows: {len(maate):,}")
maate.head()


### Cleaning: two non-canton codes

Two `DPA_CANTON` codes in this dataset are **not real cantons** and would
distort the map if left in:

- **`9999` ("MARINA")** — a placeholder for marine protected areas that don't
  belong to any single canton's land territory. At ~20.9 million hectares
  (dwarfing every actual canton's area), including it would make every real
  canton's conservation area look negligible by comparison on the color scale.
- **`8888` ("ISLA")** — a generic placeholder code, not a real canton.

We also fix one canton-boundary mismatch: code **`2302`** ("La Concordia",
recorded under Santo Domingo de los Tsáchilas province) doesn't exist in our
2011-vintage GeoJSON, because La Concordia was administratively transferred
from Esmeraldas to Santo Domingo province in **2013** — after the boundary
file was published. Its territory in the GeoJSON is still filed under its
pre-2013 code, **`0808`** (also named "La Concordia", under Esmeraldas). We
remap it rather than drop it, so the canton's conservation area isn't
silently lost from the map.


In [ ]:
# Drop the two non-canton placeholder codes
maate_clean = maate[~maate['DPA_CANTON'].isin([9999, 8888])].copy()

# Zero-pad to match the GeoJSON's 4-character string codes (e.g. 101 -> '0101')
maate_clean['DPA_CANTON'] = maate_clean['DPA_CANTON'].astype(str).str.zfill(4)

# Remap La Concordia's post-2013 code to its pre-2013 code, which is what the
# 2011-vintage boundary file actually contains.
maate_clean['DPA_CANTON'] = maate_clean['DPA_CANTON'].replace({'2302': '0808'})

# Verify every remaining canton code exists in the GeoJSON boundaries —
# an unmatched code would silently disappear from the choropleth.
geo_canton_codes = {f['properties']['DPA_CANTON'] for f in cantones_geojson['features']}
unmatched = set(maate_clean['DPA_CANTON']) - geo_canton_codes
assert not unmatched, f"Unmatched canton codes: {unmatched}"
print(f"All {maate_clean['DPA_CANTON'].nunique()} canton codes matched to GeoJSON boundaries.")


In [ ]:
geo_canton_codes = {f['properties']['DPA_CANTON'] for f in cantones_geojson['features']}
maate_codes = set(maate_clean['DPA_CANTON'])

faltantes = geo_canton_codes - maate_codes
print(f"Cantones sin dato en MAATE: {len(faltantes)}")

# Para verlos con nombre, usando el geojson
faltantes_info = [
    (f['properties']['DPA_CANTON'], f['properties']['DPA_DESCAN'], f['properties']['DPA_DESPRO'])
    for f in cantones_geojson['features']
    if f['properties']['DPA_CANTON'] in faltantes
]
for codigo, canton, provincia in sorted(faltantes_info):
    print(codigo, canton, provincia)

In [ ]:
import pandas as pd

# Universo completo de cantones (código + nombre) desde el geojson
geo_cantones = pd.DataFrame([
    {
        'DPA_CANTON': f['properties']['DPA_CANTON'],
        'DPA_DESCAN': f['properties']['DPA_DESCAN'],
        'DPA_DESPRO': f['properties']['DPA_DESPRO'],
    }
    for f in cantones_geojson['features']
])

# Suponiendo que ya tienes el área agregada por cantón, p. ej.:
# area_por_canton = maate_clean.groupby('DPA_CANTON', as_index=False)['AREA_HA'].sum()
area_por_canton = maate_clean.groupby('DPA_CANTON', as_index=False)['AREA_HA'].sum()

# Merge completo: todos los 224 cantones, con 0 donde no hay dato
choropleth_data = geo_cantones.merge(area_por_canton[['DPA_CANTON', 'AREA_HA']],
                                      on='DPA_CANTON', how='left')
choropleth_data['AREA_HA'] = choropleth_data['AREA_HA'].fillna(0)

print(f"Cantones en el choropleth: {choropleth_data['DPA_CANTON'].nunique()} (deberían ser 224)")
print(f"Cantones con 0 hectáreas: {(choropleth_data['AREA_HA'] == 0).sum()}")

In [ ]:
# Aggregate to one total hectares figure per canton (a canton can have
# multiple rows: one per parish, and/or one per conservation category).
conservation_by_canton = (
    maate_clean.groupby(['DPA_CANTON', 'DPA_DESCAN'])['AREA_HA']
    .sum()
    .reset_index()
)

print(f"Cantons with conservation area on record: {len(conservation_by_canton)}")
conservation_by_canton.sort_values('AREA_HA', ascending=False).head(10)


In [ ]:
fig_choro = px.choropleth(
    conservation_by_canton,
    geojson=cantones_geojson,
    locations='DPA_CANTON',
    featureidkey='properties.DPA_CANTON',
    color='AREA_HA',
    color_continuous_scale='Greens',   # sequential: one hue, more-is-darker
    hover_name='DPA_DESCAN',
    labels={'AREA_HA': 'Conservation area (ha)'},
)
fig_choro.update_geos(fitbounds='locations', visible=False)
fig_choro.update_layout(
    title='Surface Area Under Conservation by Canton — Ecuador, December 2022<br>'
          '<sub>Sequential color scale: darker green = more hectares under conservation</sub>',
    margin=dict(l=0, r=0, t=60, b=0), height=650,
)
fig_choro.show()


### Reading the choropleth

The darkest cantons cluster in the **Amazon region** (Napo, Pastaza,
Sucumbíos, Orellana, Morona Santiago) — vast, sparsely populated cantons that
contain large stretches of protected rainforest, national parks, and Socio
Bosque conservation agreements. Coastal and highland cantons, which tend to be
smaller and more densely populated with land under agriculture or urban use,
show far less conservation area in absolute hectares.

Per the lecture's choropleth recommendations, this map is honest **because the
variable itself — total hectares under conservation — genuinely is a
per-region quantity**, not something that scales with each canton's
population or land area in a way that would need normalizing. (Contrast this
with Part 2's next dataset, where normalizing by population turns out to
matter a great deal.)


## 10. Proportional Symbol Map & Dot Density: Detentions/Apprehensions (MDI)

Our second spatial dataset comes from Ecuador's **Ministry of the Interior**:
individual-level records of detentions and apprehensions nationwide, January
through June 2026. Unlike the MAATE data, this file gives us something rarer —
**exact latitude/longitude for every single incident**, not just a canton
code. That lets us build two different idioms from the *same* underlying data:
a proportional symbol map (aggregated by canton) and a genuine dot density map
(one dot per real incident location, not simulated).

As with MAATE's `AREA_HA` column, `latitud` and `longitud` are stored as text
with a **comma decimal separator** (`'-2,293860393'`) — the same
Spanish-locale export convention, so we convert both columns to numbers
explicitly right after loading.


In [ ]:
MDI_URL = ('https://raw.githubusercontent.com/CarlaGeovanna/'
           'datasets_public/main/mdi_detenidosaprehendidos_pm_2026_enero_junio.xlsx')

# Excel files can't be streamed row-by-row like a CSV url, so we download the
# raw bytes first and read the workbook from memory.
import io

with urllib.request.urlopen(MDI_URL) as response:
    mdi_bytes = response.read()

mdi = pd.read_excel(io.BytesIO(mdi_bytes), sheet_name='1. Detenidos y Aprehendidos')

# latitud/longitud are stored as TEXT with a comma decimal separator
# (e.g. '-2,293860393'), the same Spanish-locale export convention we saw in
# the MAATE file's AREA_HA column — pandas reads them as strings until we
# convert them explicitly.
mdi['latitud'] = mdi['latitud'].astype(str).str.replace(',', '.').astype(float)
mdi['longitud'] = mdi['longitud'].astype(str).str.replace(',', '.').astype(float)

print(f"Rows: {len(mdi):,}")
mdi[['tipo', 'edad', 'sexo', 'codigo_provincia', 'nombre_provincia',
     'codigo_canton', 'nombre_canton', 'presunta_infraccion',
     'latitud', 'longitud']].head()

### Cleaning: the same canton-code issues, plus one new one

`codigo_canton` needs the same two fixes as the MAATE data above (the La
Concordia boundary remap), plus one MDI-specific case: code **`0`**
("MAR TERRITORIAL") marks incidents that happened in **territorial waters**,
outside any canton's land boundary — real, valid incidents with real
coordinates, just not attributable to a specific canton. We keep these for the
dot density map (which only needs lat/lon) but exclude them from the
canton-aggregated symbol map (which needs a valid canton to attach to).


In [ ]:
# codigo_canton is numeric in the source file; zero-pad to match GeoJSON codes.
mdi['codigo_canton'] = mdi['codigo_canton'].astype(str).str.zfill(4)

# Same 2013 boundary remap as the MAATE dataset (La Concordia).
mdi['codigo_canton'] = mdi['codigo_canton'].replace({'2302': '0808'})

# Territorial-water incidents (code '0000') have no canton — valid for the
# dot density map (has real coordinates) but excluded from canton aggregation.
mdi_land = mdi[mdi['codigo_canton'] != '0000'].copy()

unmatched = set(mdi_land['codigo_canton']) - geo_canton_codes
assert not unmatched, f"Unmatched canton codes: {unmatched}"
print(f"Land-based incidents: {len(mdi_land):,} / {len(mdi):,} total")
print(f"Territorial-water incidents (kept for dot map, excluded from canton totals): "
      f"{(mdi['codigo_canton'] == '0000').sum()}")


### Choosing the color channel

`presunta_infraccion` (alleged offense category) has 46 distinct values —
far too many for direct color encoding (recall the course's color methodology:
a scatter/bubble map's all-pairs colorblind-safety check caps out at **4**
categorical slots). We group the offense categories into the three largest
plus an "Otro" bucket, exactly as we did with accident types in the traffic
fatality symbol map earlier this course.


In [ ]:
print(mdi_land['presunta_infraccion'].value_counts().head(6))


In [ ]:
top_infractions = [
    'DELITOS CONTRA EL DERECHO A LA PROPIEDAD',
    'DELITOS POR LA PRODUCCIÓN O TRÁFICO ILÍCITO DE SUSTANCIAS CATALOGADAS SUJETAS A FISCALIZACIÓN',
    'BOLETAS',
]
mdi_land['infraccion_agrupada'] = mdi_land['presunta_infraccion'].where(
    mdi_land['presunta_infraccion'].isin(top_infractions), 'Otro'
)

infraccion_colors = {
    top_infractions[0]: '#2a78d6',  # blue
    top_infractions[1]: '#008300',  # green
    top_infractions[2]: '#e87ba4',  # magenta
    'Otro':              '#eda100', # yellow
}
infraccion_order = top_infractions + ['Otro']
infraccion_labels = {
    top_infractions[0]: 'Contra la propiedad',
    top_infractions[1]: 'Tráfico de sustancias',
    top_infractions[2]: 'Boletas (orden de captura)',
    'Otro': 'Otro',
}
mdi_land['infraccion_label'] = mdi_land['infraccion_agrupada'].map(infraccion_labels)


In [ ]:
# Aggregate to one row per (canton, offense group) — one circle per row.
agg_mdi = (
    mdi_land.groupby(['codigo_canton', 'nombre_canton', 'infraccion_label'])
    .size()
    .reset_index(name='Casos')
)

# Attach each canton's centroid for plotting. Ecuador's cantons don't ship
# with pre-computed centroids in this GeoJSON, so we approximate each
# canton's anchor point as the mean lat/lon of its incidents — good enough
# for a symbol map (unlike Section 9's choropleth, which uses the true
# polygon boundaries directly and needs no point approximation at all).
canton_centroid = (
    mdi_land.groupby('codigo_canton')[['latitud', 'longitud']]
    .mean()
    .rename(columns={'latitud': 'lat', 'longitud': 'lon'})
)
agg_mdi = agg_mdi.merge(canton_centroid, on='codigo_canton', how='left')

print(f"Circles on the symbol map: {len(agg_mdi)}")
agg_mdi.sort_values('Casos', ascending=False).head(8)


In [ ]:
import json

features = [
    {
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [row.lon, row.lat]  # GeoJSON = [lon, lat], no [lat, lon]
        },
        "properties": {
            "codigo_canton": row.codigo_canton,
            "nombre_canton": row.nombre_canton,
            "infraccion_label": row.infraccion_label,
            "Casos": int(row.Casos)
        }
    }
    for row in agg_mdi.itertuples(index=False)
]

agg_mdi_geojson = {"type": "FeatureCollection", "features": features}

with open('agg_mdi.geojson', 'w', encoding='utf-8') as f:
    json.dump(agg_mdi_geojson, f, ensure_ascii=False)

print(f"GeoJSON exportado: {len(features)} features -> agg_mdi.geojson")

In [ ]:
fig_symbol = px.scatter_geo(
    agg_mdi,
    lat='lat', lon='lon',
    size='Casos',
    color='infraccion_label',
    category_orders={'infraccion_label': list(infraccion_labels.values())},
    color_discrete_map={v: infraccion_colors[k] for k, v in infraccion_labels.items()},
    hover_name='nombre_canton',
    hover_data={'infraccion_label': True, 'Casos': True, 'lat': False, 'lon': False},
    size_max=40, opacity=0.7,
    scope='south america',
)
fig_symbol.update_geos(
    lataxis_range=[-6, 2], lonaxis_range=[-92, -74],
    showcountries=True, countrycolor='#c3c2b7',
    showland=True, landcolor='#f9f9f7',
    showocean=True, oceancolor='#eaf2fb', coastlinecolor='#898781',
)
fig_symbol.update_traces(marker=dict(line=dict(width=0.4, color='white')))
fig_symbol.update_layout(
    title='Detentions/Apprehensions by Canton — Jan–Jun 2026<br>'
          '<sub>Circle size = total cases · Circle color = offense category</sub>',
    legend_title_text='Offense category',
    margin=dict(l=0, r=0, t=70, b=0), height=650,
)
fig_symbol.show()


In [ ]:
fig_symbol = px.scatter_map(
    agg_mdi,
    lat='lat', lon='lon',
    size='Casos',
    color='infraccion_label',
    category_orders={'infraccion_label': list(infraccion_labels.values())},
    color_discrete_map={v: infraccion_colors[k] for k, v in infraccion_labels.items()},
    hover_name='nombre_canton',
    hover_data={'infraccion_label': True, 'Casos': True, 'lat': False, 'lon': False},
    size_max=40, opacity=0.7,
    zoom=5.2,
    center={'lat': -1.5, 'lon': -78.5},  # centro aprox. de Ecuador continental
    map_style='open-street-map',         # tiles reales de OSM, sin token
)
fig_symbol.update_traces(marker=dict(opacity=0.75))
fig_symbol.update_layout(
    title='Detentions/Apprehensions by Canton — Jan–Jun 2026<br>'
          '<sub>Circle size = total cases · Circle color = offense category</sub>',
    legend_title_text='Offense category',
    margin=dict(l=0, r=0, t=70, b=0), height=650,
)
fig_symbol.show()

## 11. Dot Density Map: Individual Incident Locations

Because this dataset has real per-incident coordinates, we can build the
idiom the lecture describes as visualizing "distribution of a phenomenon by
placing dots" — but with a twist. The lecture's own example map (a
population-density map of the US) has a legend reading "1 Dot = 50,000
Population": each dot there stands in for a **fixed count of people**, not a
single real observation, because census data is normally only available
pre-aggregated by region, not as one row per person. That 50,000 figure is
just what that particular map's data required — a dot density map's scale is
always chosen per-dataset, not a fixed convention. Here we can do better than
even that: our MDI data is already at the individual-incident level, so **each
dot is one real incident** at its true recorded location — no aggregation, no
per-dot scale to choose, no simulation needed.


In [ ]:
fig_dot = px.scatter_geo(
    mdi,  # full dataset, including the 48 territorial-water incidents
    lat='latitud', lon='longitud',
    scope='south america',
    opacity=0.35,
)
fig_dot.update_traces(marker=dict(size=3, color='#2a78d6'))
fig_dot.update_geos(
    lataxis_range=[-6, 2], lonaxis_range=[-92, -74],
    showcountries=True, countrycolor='#c3c2b7',
    showland=True, landcolor='#f9f9f7',
    showocean=True, oceancolor='#eaf2fb', coastlinecolor='#898781',
)
fig_dot.update_layout(
    title='Dot Density Map — Every Detention/Apprehension, Jan–Jun 2026<br>'
          '<sub>One dot = one real incident (exact recorded coordinates)</sub>',
    margin=dict(l=0, r=0, t=60, b=0), height=650,
)
fig_dot.show()


In [ ]:
fig_dot = px.scatter_map(
    mdi,  # full dataset, including the 48 territorial-water incidents
    lat='latitud', lon='longitud',
    zoom=5.2,
    center={'lat': -1.5, 'lon': -78.5},
    map_style='open-street-map',
    opacity=0.35,
)
fig_dot.update_traces(marker=dict(size=3, color='#2a78d6'))

# --- Botones para alternar entre varios mapas base ---
base_maps = [
    ('OpenStreetMap', {'map.style': 'open-street-map', 'map.layers': []}),
    ('Carto Claro',   {'map.style': 'carto-positron',   'map.layers': []}),
    ('Carto Oscuro',  {'map.style': 'carto-darkmatter',  'map.layers': []}),
    ('Satelital (Esri)', {
        'map.style': 'white-bg',
        'map.layers': [{
            'below': 'traces',
            'sourcetype': 'raster',
            'sourceattribution': 'Esri, World Imagery',
            'source': [
                'https://server.arcgisonline.com/ArcGIS/rest/services/'
                'World_Imagery/MapServer/tile/{z}/{y}/{x}'
            ],
        }],
    }),
    ('Calles (Esri)', {
        'map.style': 'white-bg',
        'map.layers': [{
            'below': 'traces',
            'sourcetype': 'raster',
            'sourceattribution': 'Esri, World Street Map',
            'source': [
                'https://server.arcgisonline.com/ArcGIS/rest/services/'
                'World_Street_Map/MapServer/tile/{z}/{y}/{x}'
            ],
        }],
    }),
]

fig_dot.update_layout(
    updatemenus=[dict(
        type='buttons', direction='right',
        buttons=[dict(label=label, method='relayout', args=[args]) for label, args in base_maps],
        x=0.01, y=0.02, xanchor='left', yanchor='bottom',
        bgcolor='white', bordercolor='#c3c2b7', font=dict(size=11),
    )],
    title='Dot Density Map — Every Detention/Apprehension, Jan–Jun 2026<br>'
          '<sub>One dot = one real incident (exact recorded coordinates)</sub>',
    margin=dict(l=0, r=0, t=60, b=0), height=650,
)
fig_dot.show()

### Reading the dot density map

The densest clusters sit directly on **Quito** and **Guayaquil** — which,
per the lecture's "Beware: population maps trickiness" warning, is exactly
what you'd expect from *any* map of a phenomenon that scales with population
density, regardless of what the phenomenon actually is. A dense cluster over a
city center does not, by itself, tell us whether that city has a
disproportionately *high rate* of incidents — it may simply have more people.
This is precisely the trap the next section addresses directly.


### Finding Clusters Automatically — By Incident Profile, Not Location

The previous map already answered *where* incidents concentrate. Now we ask a
genuinely different question: does the **same kind** of incident — the same
mix of offense, place type, mobilization, and timing — show up as a
repeatable pattern in many different parts of the country, or is every
region's mix of incidents unique to that region? Answering that requires
clustering on **incident profile**, not on location — so this time we run
DBSCAN on a feature set that deliberately excludes every column that names or
encodes a place.


**Final feature set:**
- `infraccion_agrupada` — the four-bucket offense grouping we already built
  in Section 10 (reused as-is, not rebuilt)
- `tipo_lugar` — category of place
- `movilizacion` — how the person was moving (on foot, by car, by
  motorcycle, ...)
- `tipo_arma` — weapon category present, if any
- `hour_of_day` — derived from `hora_detencion_aprehension`; a behavioral
  signal (time of day), not a spatial one
- `day_of_week` — derived from `fecha_detencion_aprehension`; same reasoning




In [ ]:
# ColumnTransformer/OneHotEncoder/StandardScaler are all from sklearn, so one
# defensive try/except covers everything this cell needs (same pattern as
# the DBSCAN import above).
try:
    from sklearn.cluster import DBSCAN
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder, StandardScaler
except ImportError:
    !pip install scikit-learn
    from sklearn.cluster import DBSCAN
    from sklearn.compose import ColumnTransformer
    from sklearn.preprocessing import OneHotEncoder, StandardScaler

import numpy as np

# Two behavioral time features, derived from the timestamp columns —
# when an incident happened, not where.
mdi_land['hour_of_day'] = pd.to_datetime(
    mdi_land['hora_detencion_aprehension'].astype(str), format='%H:%M:%S'
).dt.hour
mdi_land['day_of_week'] = mdi_land['fecha_detencion_aprehension'].dt.dayofweek

categorical_features = ['infraccion_agrupada', 'tipo_lugar', 'movilizacion', 'tipo_arma']
numeric_features = ['hour_of_day', 'day_of_week']

# One-hot encode the categoricals, standard-scale the numerics, and
# concatenate the result into a single numeric matrix DBSCAN can use.
encoder = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ('num', StandardScaler(), numeric_features),
])
X_profile = encoder.fit_transform(mdi_land[categorical_features + numeric_features])
print(f"Encoded feature matrix shape: {X_profile.shape}")

EPS_PROFILE = 0.7        # Euclidean distance in standardized/one-hot space —
                          # no physical unit; chosen empirically
MIN_SAMPLES_PROFILE = 30  # an incident profile needs at least 30 similar
                           # neighbors to count as a recurring pattern

db_profile = DBSCAN(eps=EPS_PROFILE, min_samples=MIN_SAMPLES_PROFILE, metric='euclidean')
mdi_land['cluster'] = db_profile.fit_predict(X_profile)

n_clusters = mdi_land['cluster'].nunique() - (1 if -1 in mdi_land['cluster'].values else 0)
n_noise = (mdi_land['cluster'] == -1).sum()
print(f"Profile clusters found: {n_clusters}")
print(f"Incidents with no similar-profile neighbors (noise): "
      f"{n_noise:,} ({100 * n_noise / len(mdi_land):.1f}% of all incidents)")

# Evidence for the claim in the next markdown cell: are these profile
# clusters confined to one place, or do they recur nationally? For each of
# the five largest clusters, count how many of Ecuador's 24 provinces its
# points touch.
top5 = mdi_land['cluster'].value_counts().drop(-1, errors='ignore').head(5)
dispersion = pd.DataFrame({
    'cluster': top5.index,
    'incidents': top5.values,
    'provinces_touched': [
        mdi_land.loc[mdi_land['cluster'] == cid, 'nombre_provincia'].nunique()
        for cid in top5.index
    ],
})
print("\nTop 5 clusters by size — how many of Ecuador's 24 provinces each touches:")
dispersion


### Labeling Each Point with Its Canton (Hover Context Only)

The clustering never used geographic data — it grouped points purely by incident profile. Once mapped, we simply attach each point's canton name for hover context, using the same point-in-polygon join as before (via Shapely's STRtree) — purely descriptive, since it plays no role in defining the clusters themselves.

In [ ]:
from shapely.geometry import shape
from shapely import points as shapely_points
from shapely.strtree import STRtree

canton_polygons = [shape(f['geometry']) for f in cantones_geojson['features']]
canton_names_list = [f['properties']['DPA_DESCAN'] for f in cantones_geojson['features']]
canton_tree = STRtree(canton_polygons)

# Vectorized point-in-polygon: build one Shapely Point per incident, then ask
# the tree which polygon each point falls inside (predicate='intersects').
# Returns two parallel arrays: which point index matched which polygon index.
# Runs on mdi_land (not the full mdi) so canton_name stays row-aligned with
# the cluster column built above, which was also fit on mdi_land.
incident_points = shapely_points(mdi_land['longitud'].values, mdi_land['latitud'].values)
matched_point_idx, matched_poly_idx = canton_tree.query(incident_points, predicate='intersects')

mdi_land['canton_name'] = None
mdi_land.iloc[matched_point_idx, mdi_land.columns.get_loc('canton_name')] = [
    canton_names_list[i] for i in matched_poly_idx
]
mdi_land['canton_name'] = mdi_land['canton_name'].fillna('Offshore / unmatched')

n_unmatched = (mdi_land['canton_name'] == 'Offshore / unmatched').sum()
print(f"Incidents matched to a canton: {len(mdi_land) - n_unmatched:,} / {len(mdi_land):,} "
      f"({100 * n_unmatched / len(mdi_land):.1f}% offshore or on a boundary edge)")

### Highlighting One Cluster at a Time

With the canton name now attached to every point, we build the profile-cluster map with the same interaction pattern as the geographic version: no boundary lines, and each cluster drawn as its own trace so hovering can isolate exactly one cluster.

The key difference to watch for: because these clusters are defined by incident profile, not location, a single cluster's points should scatter across many cantons and provinces, rather than sitting in one physical neighborhood the way the geographic hotspots did.

In [ ]:
mdi_clustered = mdi_land[mdi_land['cluster'] != -1].copy()
mdi_noise = mdi_land[mdi_land['cluster'] == -1]

fig_clusters = go.Figure()

fig_clusters.add_trace(go.Scattergeo(
    lat=mdi_noise['latitud'], lon=mdi_noise['longitud'],
    mode='markers', marker=dict(size=2, color='#c3c2b7'), opacity=0.25,
    name='No cluster (noise)', hoverinfo='skip',
))

# One trace PER CLUSTER (not one shared trace) — this is what lets hovering
# over a cluster highlight only that cluster's own points, since Plotly
# dims every trace except the one under the cursor.
cluster_ids = sorted(mdi_clustered['cluster'].unique())
palette = px.colors.sample_colorscale('Turbo', [i / max(len(cluster_ids) - 1, 1) for i in range(len(cluster_ids))])

for color, cid in zip(palette, cluster_ids):
    sub = mdi_clustered[mdi_clustered['cluster'] == cid]
    top_canton = sub['canton_name'].mode().iat[0]  # most common canton within this cluster
    top_infraccion = sub['infraccion_agrupada'].mode().iat[0]  # this cluster's dominant profile trait
    fig_clusters.add_trace(go.Scattergeo(
        lat=sub['latitud'], lon=sub['longitud'],
        mode='markers',
        marker=dict(size=4, color=color),
        opacity=0.7,
        hovertext=(f"Cluster {cid} — {len(sub)} incidents<br>"
                   f"Most common offense: {top_infraccion}<br>"
                   f"Example canton: {top_canton}"),
        hoverinfo='text',
        name=f'Cluster {cid}', showlegend=False,
    ))

fig_clusters.update_geos(
    scope='south america',
    lataxis_range=[-6, 2], lonaxis_range=[-92, -74],
    showcountries=True, countrycolor='#c3c2b7',
    showland=True, landcolor='#f9f9f7',
    showocean=True, oceancolor='#eaf2fb', coastlinecolor='#898781',
)
fig_clusters.update_layout(
    title='Detentions/Apprehensions — Clusters by Incident Profile<br>'
          f'<sub>{n_clusters} profile clusters detected from non-geographic features · '
          f'gray = no similar-profile neighbors · '
          f'hover a cluster to isolate it and see its dominant profile</sub>',
    margin=dict(l=0, r=0, t=70, b=0), height=700,
)
fig_clusters.show()


### Reading the profile-clustered map

Profile-based clustering (not geographic) shows that the same types of incidents recur across nearly every province in the country, rather than concentrating in a single city. This contrasts with the earlier geographic map, which mostly just showed where more people live (Quito/Guayaquil) — profile clustering reveals genuine national patterns (e.g., "on foot, midday, non-specific offense") that aren't explained by population density, but by the type of incident itself.



## 12. Why Normalization Matters: Raw Counts vs. Rate per Province

The lecture's xkcd example makes the point sharply: a map of almost *any*
absolute count mostly just re-draws a population map, because more people
means more of everything. We can demonstrate this directly with our own data
by comparing two views of the *same* province-level totals: **raw case
counts** vs. **cases normalized by population**.

We use the population figures already embedded in `ecuador.geojson`
(`pob_tot`, from the 2010 census — still a reasonable population-weighting
proxy for this comparison, since province-level population ranks change
slowly).


In [ ]:
PROVINCIAS_GEOJSON_URL = ('https://raw.githubusercontent.com/CarlaGeovanna/'
                           'datasets_public/refs/heads/main/ecuador.geojson')

with urllib.request.urlopen(PROVINCIAS_GEOJSON_URL) as response:
    provincias_geojson = json.load(response)

# Build a province population lookup: dpa_provin code -> total population
prov_pop = {
    f['properties']['dpa_provin']: f['properties']['pob_tot']
    for f in provincias_geojson['features']
}

# codigo_provincia in the MDI data is already 2-digit zero-padded, matching
# dpa_provin directly — no remapping needed here.
mdi['codigo_provincia'] = mdi['codigo_provincia'].astype(str).str.zfill(2)

raw_counts = mdi.groupby(['codigo_provincia', 'nombre_provincia']).size().reset_index(name='Casos')
raw_counts['poblacion'] = raw_counts['codigo_provincia'].map(prov_pop)
raw_counts = raw_counts.dropna(subset=['poblacion'])  # drop provinces absent from the 2010 file, if any

# Rate per 100,000 inhabitants — the same normalization convention the
# lecture recommends ("unemployed people per 100 citizens").
raw_counts['tasa_100k'] = raw_counts['Casos'] / raw_counts['poblacion'] * 100_000

raw_counts.sort_values('Casos', ascending=False).head(8)

In [ ]:
fig_norm = go.Figure()
fig_norm.add_trace(go.Bar(
    y=raw_counts.sort_values('Casos')['nombre_provincia'],
    x=raw_counts.sort_values('Casos')['Casos'],
    orientation='h', marker_color='#2a78d6', name='Casos totales (crudo)',
))
fig_norm.update_layout(
    title='Raw Case Counts by Province — Dominated by Population Size',
    xaxis_title='Total cases', height=650, margin=dict(l=0, r=20, t=60, b=0),
)
fig_norm.show()


In [ ]:
fig_rate = go.Figure()
sorted_by_rate = raw_counts.sort_values('tasa_100k')
fig_rate.add_trace(go.Bar(
    y=sorted_by_rate['nombre_provincia'],
    x=sorted_by_rate['tasa_100k'],
    orientation='h', marker_color='#e34948', name='Tasa por 100k',
))
fig_rate.update_layout(
    title='Cases per 100,000 Inhabitants by Province — Normalized',
    xaxis_title='Cases per 100,000 population', height=650, margin=dict(l=0, r=20, t=60, b=0),
)
fig_rate.show()


### The normalization effect

In the **raw count** chart, **Guayas** and **Pichincha** dominate — no
surprise, since they are Ecuador's two most populous provinces by a wide
margin. But in the **normalized rate** chart, the ranking reshuffles: some
smaller provinces climb into the highest positions once we account for how
few people live there, while Guayas and Pichincha — still large in absolute
terms — drop toward the middle of the pack once expressed as a rate.

This is the exact failure mode the lecture warns about: **"most attributes
just show where people live."** A raw-count map of almost any per-person
phenomenon in Ecuador would visually resemble a population density map,
because population size is the single strongest driver of almost every count.
Reporting the *rate* instead answers a genuinely different — and usually more
useful — question: not "where did the most incidents happen," but "where is a
resident most likely to be involved in one."


## 13. Map Projections: The Same Data, Different Distortion

Every map so far has silently chosen a projection without discussing it. Since a sphere can't be flattened without distortion, every projection sacrifices something — area, shape, distance, or direction. Mercator (the most common) preserves angles but heavily distorts area near the poles. Next, we'll see that distortion at global scale, then check whether it actually matters for Ecuador-only maps.


In [ ]:
# No new dataset needed here — Plotly's Scattergeo can render coastlines and
# country borders directly from its own built-in world atlas. We plot two
# empty (invisible) marker traces just to give each subplot a map to draw on,
# then style the land/country layer itself, which is what actually carries
# the projection's distortion.
from plotly.subplots import make_subplots
import plotly.graph_objects as go

projections = ['mercator', 'natural earth']
fig_world = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'scattergeo'}, {'type': 'scattergeo'}]],
    subplot_titles=[p.title() for p in projections],
)

for i in range(2):
    fig_world.add_trace(
        go.Scattergeo(lat=[0], lon=[0], mode='markers',
                       marker=dict(size=0.1, opacity=0), showlegend=False),
        row=1, col=i + 1,
    )

fig_world.update_geos(
    showcountries=True, countrycolor='#898781',
    showland=True, landcolor='#f2c94c',
    showocean=True, oceancolor='#eaf2fb',
    showcoastlines=True, coastlinecolor='#555',
    lataxis_showgrid=True, lonaxis_showgrid=True,
)
for i, proj in enumerate(projections):
    fig_world.update_geos(projection_type=proj, row=1, col=i + 1)

fig_world.update_layout(
    title='Same World, Two Projections — Compare Greenland to Africa',
    height=550, margin=dict(l=0, r=0, t=80, b=0),
)
fig_world.show()


### Reading the world comparison

In Mercator, Greenland appears nearly as large as Africa, even though Africa is actually ~14x bigger — because Mercator's distortion grows with distance from the equator. Natural Earth keeps landmasses closer to their true relative size instead. Since distortion depends on latitude (not size), a country sitting on the equator, like Ecuador, should barely be distorted at all — which we'll verify next with our own data.


In [ ]:
# Reuse Section 12's raw_counts (with tasa_100k already computed) and
# provincias_geojson — no new data, just the same choropleth rendered three
# times with a different projection_type each time.
projections_ec = ['mercator', 'orthographic', 'natural earth']
fig_ec_proj = make_subplots(
    rows=1, cols=3,
    specs=[[{'type': 'choropleth'}] * 3],
    subplot_titles=[p.title() for p in projections_ec],
)

for i, proj in enumerate(projections_ec):
    fig_ec_proj.add_trace(
        go.Choropleth(
            geojson=provincias_geojson,
            locations=raw_counts['codigo_provincia'],
            z=raw_counts['tasa_100k'],
            featureidkey='properties.dpa_provin',
            colorscale='Oranges',
            showscale=(i == 2),
            colorbar=dict(title='Cases per<br>100k'),
        ),
        row=1, col=i + 1,
    )

fig_ec_proj.update_geos(fitbounds='locations', visible=False)
for i, proj in enumerate(projections_ec):
    fig_ec_proj.update_geos(projection_type=proj, row=1, col=i + 1)

fig_ec_proj.update_layout(
    title='Detentions/Apprehensions Rate by Province — Same Data, Three Projections',
    height=500, margin=dict(l=0, r=0, t=80, b=0),
)
fig_ec_proj.show()


### Reading the Ecuador comparison


Unlike the world map, these three Ecuador panels look nearly identical — as predicted, since Ecuador sits on the equator, where Mercator's distortion is minimal. The lesson: projection choice matters most for regions spanning wide latitudes or sitting far from the equator; for a small equatorial country, Mercator is a safe default.

## 14. Adding a Time Dimension: How Rates Evolved Month by Month

A static choropleth hides whether a province's rate stayed constant or spiked/settled over time — so we add time as an explicit dimension using Plotly's animation_frame, reusing Section 12's logic but grouping by month instead of collapsing all six months into one total. Key detail: the color scale range is fixed across all months (not per-frame), so colors stay comparable and don't misleadingly rescale for each frame.


In [ ]:
# Group by month AND province (instead of Section 12's province-only
# grouping) to get one row per (month, province) combination.
mdi['mes'] = mdi['fecha_detencion_aprehension'].dt.to_period('M').dt.to_timestamp()

monthly_counts = (
    mdi.groupby(['mes', 'codigo_provincia', 'nombre_provincia'])
    .size()
    .reset_index(name='Casos')
)
monthly_counts['poblacion'] = monthly_counts['codigo_provincia'].map(prov_pop)
monthly_counts = monthly_counts.dropna(subset=['poblacion'])
monthly_counts['tasa_100k'] = monthly_counts['Casos'] / monthly_counts['poblacion'] * 100_000

# animation_frame needs a clean, sortable label per period.
monthly_counts['mes_label'] = monthly_counts['mes'].dt.strftime('%Y-%m')
monthly_counts = monthly_counts.sort_values('mes')  # frame order follows row order

fig_animated = px.choropleth(
    monthly_counts,
    geojson=provincias_geojson,
    locations='codigo_provincia',
    featureidkey='properties.dpa_provin',
    color='tasa_100k',
    range_color=(0, monthly_counts['tasa_100k'].max()),  # fixed scale — see markdown above
    animation_frame='mes_label',
    color_continuous_scale='Oranges',
    hover_name='nombre_provincia',
    labels={'tasa_100k': 'Cases per 100k', 'mes_label': 'Month'},
)
fig_animated.update_geos(fitbounds='locations', visible=False)
fig_animated.update_layout(
    title='Detentions/Apprehensions Rate by Province — Monthly, Jan–Jun 2026',
    margin=dict(l=0, r=0, t=60, b=0), height=650,
)

# Slow the animation down so each month's map is on screen long enough to
# actually read (Plotly Express's default is a brisk 500ms per frame) — both
# the frame's hold time and the fade between frames need to be increased, or
# only one of the two would visibly change.
fig_animated.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1500
fig_animated.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 1000
for frame in fig_animated.frames:
    frame.layout.update(transition=dict(duration=1000))

fig_animated.show()


### Reading the animation

Sucumbíos shows the largest month-to-month swing (~72 in Jan → mid-50s in Feb → over 100 by June), while Galápagos stays elevated but stable — a distinction a static average map can't show. Nationally, monthly totals swing over 50% (5,300 in Feb to 8,200 in May). Whether these swings are a real pattern or just noise from a short window is a question only the animation can raise, not answer — but that's exactly what a static map couldn't do at all.


---

# 🎯 Activities

Both activities below come **after** all of this notebook's guided content —
Part 1 (network data) and Part 2 (spatial data) are both fully worked through
before you're asked to apply any of it yourself.


## 🎯 Activity 1: Filtering the Network by Route Density

So far we treated every route the same way, regardless of how many airlines
fly it. Some city pairs (like Quito–Guayaquil) are served by many overlapping
flights; others may have only one.

Using `pair_counts` (the per-city-pair route-entry counts you already built in
Section 6, while weighting the network for community detection):

1. Keep only the city pairs served by **more than one** route entry — this is
   your "high-traffic" sub-network. *Hint: filter `pair_counts` on
   `n_route_entries > 1`.*
2. Rebuild the node-link diagram (geographic layout, following Section 4's
   pattern) using only this filtered set of edges.
3. In a markdown cell below your map, answer: how many airports remain in this
   filtered network? Which routes disappeared, and does that match your
   intuition about which city pairs in Ecuador would have the most competing
   airlines?


In [54]:
# Activity 1, Step 1 — keep only city pairs served by more than one route entry
high_traffic_pairs = set(pair_counts.loc[pair_counts['n_route_entries'] > 1, 'pair'])

edges_hi = edges[
    edges.apply(lambda r: tuple(sorted([r['src_iata'], r['dst_iata']])) in high_traffic_pairs, axis=1)
].reset_index(drop=True)

nodes_hi_iata = set(edges_hi['src_iata']) | set(edges_hi['dst_iata'])
nodes_hi = ec_nodes[ec_nodes['iata'].isin(nodes_hi_iata)].copy()
degree_hi = pd.concat([edges_hi['src_iata'], edges_hi['dst_iata']]).value_counts()
nodes_hi['degree'] = nodes_hi['iata'].map(degree_hi)

dropped_pairs = set(pair_counts['pair']) - high_traffic_pairs
print(f"Airports remaining: {len(nodes_hi)} (down from {len(ec_nodes)})")
print(f"City pairs remaining: {len(high_traffic_pairs)} (down from {len(pair_counts)})")
print(f"Dropped city pairs ({len(dropped_pairs)}): {sorted(dropped_pairs)}")


Airports remaining: 11 (down from 14)
City pairs remaining: 14 (down from 18)
Dropped city pairs (4): [('TNW', 'UIO'), ('TNW', 'XMS'), ('TUA', 'UIO'), ('UIO', 'XMS')]


In [55]:
# Activity 1, Step 2 — rebuild the geographic node-link diagram, high-traffic edges only
coord_hi = nodes_hi.set_index('iata')[['lat', 'lon']]

edge_lat_hi, edge_lon_hi = [], []
for _, row in edges_hi.iterrows():
    edge_lat_hi += [coord_hi.loc[row['src_iata'], 'lat'], coord_hi.loc[row['dst_iata'], 'lat'], None]
    edge_lon_hi += [coord_hi.loc[row['src_iata'], 'lon'], coord_hi.loc[row['dst_iata'], 'lon'], None]

fig_hi = go.Figure()

fig_hi.add_trace(go.Scattergeo(
    lat=edge_lat_hi, lon=edge_lon_hi,
    mode='lines',
    line=dict(width=1, color='#898781'),
    hoverinfo='none',
    showlegend=False,
))

fig_hi.add_trace(go.Scattergeo(
    lat=nodes_hi['lat'], lon=nodes_hi['lon'],
    mode='markers+text',
    text=nodes_hi['iata'],
    textposition='top center',
    marker=dict(size=nodes_hi['degree'] * 4 + 6, color='#2a78d6',
                line=dict(width=0.5, color='white')),
    hovertext=nodes_hi['name'] + ' (' + nodes_hi['city'] + ') — '
              + nodes_hi['degree'].astype(str) + ' connections',
    hoverinfo='text',
    showlegend=False,
))

fig_hi.update_geos(
    lataxis_range=[-6, 2], lonaxis_range=[-92, -74],
    showcountries=True, countrycolor='#c3c2b7',
    showland=True, landcolor='#f9f9f7',
    showocean=True, oceancolor='#eaf2fb',
    coastlinecolor='#898781',
)
fig_hi.update_layout(
    title="Ecuador's High-Traffic Domestic Routes — Geographic Layout<br>"
          "<sub>Only city pairs with more than one route entry (n_route_entries &gt; 1)</sub>",
    margin=dict(l=0, r=0, t=60, b=0), height=650,
)
fig_hi.show()


### Answer

Filtering to city pairs served by more than one route entry shrinks the network from **14 airports / 18 city pairs** down to **11 airports / 14 city pairs**. Three airports drop out entirely — **Tena (TNW)**, **Tulcán (TUA)**, and **Macas (XMS)** — because every route touching them (TNW–UIO, TNW–XMS, TUA–UIO, UIO–XMS) was filed by only a single route entry.

This matches intuition about Ecuador's domestic market. The routes that survive connect the two big hubs, Quito (UIO) and Guayaquil (GYE), to each other and to secondary but still substantial destinations — Cuenca, Manta, Latacunga, Santa Rosa, Galápagos (both airports), Esmeraldas, Coca. Those city pairs carry enough passenger demand for multiple airlines to compete on the same route. The routes that disappeared connect UIO to small, remote regional airstrips (Tena, Tulcán, Macas) that plausibly support only a single carrier's service — not enough traffic for a second airline to bother entering.


## 🎯 Activity 2: A Population-Normalized Choropleth

Sections 8 and 9 built a choropleth from a genuinely per-region quantity
(conservation hectares) and a symbol map from raw incident counts. Now combine
both ideas: build a **choropleth of the detention/apprehension rate per
100,000 inhabitants, by province**, using the `raw_counts` table you already
built in Section 12.

Steps:
1. Using `raw_counts`, build a Plotly `choropleth` (like Section 9's, but at
   the **province** level, so use `provincias_geojson` and
   `featureidkey='properties.dpa_provin'` instead of the canton file).
2. Color by `tasa_100k` with a sequential color scale (pick any single hue —
   try `'Reds'` or `'Oranges'` to visually distinguish this map from Section
   8's green conservation map).
3. In a markdown cell below your map, compare it to the **raw count**
   choropleth you'd get by coloring the same map with `Casos` instead of
   `tasa_100k`. Which provinces look most different between the two versions?
   Does that match what you saw in the bar charts in Section 12?


In [56]:
fig_choro_tasa = px.choropleth(
    raw_counts,
    geojson=provincias_geojson,
    locations='codigo_provincia',
    featureidkey='properties.dpa_provin',
    color='tasa_100k',
    color_continuous_scale='Reds',   # sequential, single hue — distinct from Section 8's green
    hover_name='nombre_provincia',
    labels={'tasa_100k': 'Cases per 100k'},
)
fig_choro_tasa.update_geos(fitbounds='locations', visible=False)
fig_choro_tasa.update_layout(
    title='Detention/Apprehension Rate per 100,000 Inhabitants by Province — Jan–Jun 2026<br>'
          '<sub>Sequential color scale: darker red = higher rate, normalized for population</sub>',
    margin=dict(l=0, r=0, t=60, b=0), height=650,
)
fig_choro_tasa.show()


In [57]:
# Same map, colored by the RAW count instead — for direct comparison to the rate map above
fig_choro_raw = px.choropleth(
    raw_counts,
    geojson=provincias_geojson,
    locations='codigo_provincia',
    featureidkey='properties.dpa_provin',
    color='Casos',
    color_continuous_scale='Reds',
    hover_name='nombre_provincia',
    labels={'Casos': 'Total cases'},
)
fig_choro_raw.update_geos(fitbounds='locations', visible=False)
fig_choro_raw.update_layout(
    title='Detention/Apprehension Raw Case Counts by Province — Jan–Jun 2026<br>'
          '<sub>Same color hue as above, but NOT normalized by population</sub>',
    margin=dict(l=0, r=0, t=60, b=0), height=650,
)
fig_choro_raw.show()


### Answer

The two maps diverge sharply. In **raw counts**, Guayas (8,314 cases) and Pichincha (6,882) paint the darkest — they simply dominate because they're Ecuador's two most populous provinces — with Los Ríos, Manabí, and El Oro rounding out the rest of the dark cluster.

Once normalized to **cases per 100k**, those same provinces fade toward the middle of the pack (Guayas: 228/100k, Pichincha: 267/100k), while several sparsely populated Amazon and border provinces jump to the top of the color scale instead: Pastaza (658/100k), Sucumbíos (630/100k), Galápagos (609/100k), Napo (601/100k), and Morona Santiago (573/100k) — each 2–4x Guayas's rate despite having a small fraction of its raw case count. Galápagos is the most extreme case: only 153 total incidents, but because so few people live there it still lands among the highest rates in the country.

This is exactly the pattern the Section 12 bar charts already showed: the raw-count map mostly just re-draws where people live, while the rate map reveals which provinces actually have a disproportionately high incident rate relative to their population.


## 🎯 Activity 3: Community-Colored Adjacency Matrix for Ecuador's Network

Section 7 built **two** different adjacency matrices: a degree-ordered one
for Ecuador's small domestic network (`G`), and a community-colored one for
the much larger South America network (`G_sa`). The community-coloring
trick was only ever applied to the big network — `G`'s matrix stayed
ordered by degree alone, with no community information shown.

Using `G`, the `communities` you already detected with Louvain in Section
6, and the `community_of`/`community_colors` you already built there:

1. Build a node ordering that groups airports by **community first**, then
   by **degree within each community** — same idea as the South America
   version's `community_order`, but applied to `G` instead of `G_sa`.
2. Build the adjacency matrix with `nx.to_pandas_adjacency(G,
   nodelist=your_order)`, then recolor it so each filled cell inherits its
   **row airport's community color**, exactly as Section 7 did for South
   America, and render it with `px.imshow`.
3. In a markdown cell below your matrix, compare this community-colored
   version to Section 7's original degree-only matrix for the same network
   (`G`). Does the community coloring reveal anything the plain
   degree-ordered version didn't, given that the network only has three
   communities and 14 airports total?


In [58]:
# Activity 3, Step 1 — order by community first, then by degree within each community
degree_G = dict(G.degree())
community_order = sorted(
    G.nodes(),
    key=lambda n: (community_of[n], -degree_G[n])
)

# Step 2 — adjacency matrix with that ordering, recolored so each filled cell
# inherits its ROW airport's community color (same trick Section 7 used for G_sa)
adj_G_comm = nx.to_pandas_adjacency(G, nodelist=community_order)

n_comm = len(communities)
adj_G_colored = adj_G_comm.copy()
for i, node in enumerate(community_order):
    adj_G_colored.iloc[i] = adj_G_colored.iloc[i] * (community_of[node] + 1)

fig_matrix_G_comm = px.imshow(
    adj_G_colored,
    color_continuous_scale=[[0, '#fcfcfb']] + [
        [(_i + 1) / n_comm, community_colors[str(_i)]]
        for _i in range(n_comm)
    ],
    aspect='equal',
)
fig_matrix_G_comm.update_layout(
    title="Ecuador's Domestic Flight Network — Adjacency Matrix Colored by Community<br>"
          "<sub>Rows/columns grouped by Louvain community, then by degree within each — "
          "compare to Section 7's degree-only matrix for the same network</sub>",
    coloraxis_showscale=False,
    xaxis_title='', yaxis_title='',
    height=600, width=650,
)
fig_matrix_G_comm.show()


### Answer

With only 14 airports and 3 communities, the community-colored matrix mostly confirms what the degree-ordered matrix already showed near its top-left corner (UIO's and GYE's dense rows/columns) — but it adds one thing the plain degree ordering couldn't: **which specific low/mid-degree airports belong to which hub**.

In the degree-only matrix from Section 7, CUE, LOH, ESM, GPS, and SCY (all linked into the GYE side) and ETR, MEC, TNW, TUA, and XMS (all linked into the UIO side) are indistinguishable — just "less-connected airports" ordered by degree with no visual cue separating the two groups. The community coloring pulls them apart into two visibly distinct colored blocks along the diagonal (one per hub), plus a third, tiny two-airport block for **LTX–OCC**, which sits off on its own — a pair that connects only to each other and nothing else in the network. That third community is easy to miss by degree alone (both airports have degree 2, nothing that stands out), but the coloring makes it immediately obvious as its own structurally separate group.


## 🎯 Activity 4: A Point Density Heatmap

Section 11's dot density map plots all ~38,000 incidents as individual
points, but at that volume many points overlap and the densest areas just
look like solid blobs — you can't tell whether a "solid" patch has 50
incidents or 5,000. A **density heatmap** solves this differently: instead
of drawing one mark per incident, it estimates a smooth *intensity surface*
from the point locations themselves, so density differences remain visible
even where individual points would fully overlap.

Using `mdi_land`'s `latitud`/`longitud` columns:

1. Build a density heatmap with `px.density_map` (`lat=`, `lon=`,
   `radius=`, plus `map_style='open-street-map'` so the country's outline
   is visible underneath). Center the map on continental Ecuador and pick a
   `zoom` level that shows the whole country.
2. Try at least two different `radius` values. What changes visually, and
   why does a radius that's too large make the map *less* informative
   rather than more?
3. In a markdown cell below your map, compare what this idiom shows you
   about Quito and Guayaquil versus what Section 11's dot density map and
   Section 9's choropleth each showed about the same two cities. Are all
   three telling the same story, or does the density heatmap reveal
   something the other two idioms don't?


In [59]:
# Activity 4, Step 1 — density heatmap, radius=10
fig_heat_r10 = px.density_map(
    mdi_land,
    lat='latitud', lon='longitud',
    radius=10,
    center={'lat': -1.5, 'lon': -78.5}, zoom=5.4,
    map_style='open-street-map',
)
fig_heat_r10.update_layout(
    title='Density Heatmap of Detentions/Apprehensions — radius=10<br>'
          '<sub>Smooth intensity surface estimated from ~38,000 individual incident locations</sub>',
    margin=dict(l=0, r=0, t=60, b=0), height=650,
)
fig_heat_r10.show()


In [60]:
# Activity 4, Step 2 — same map, radius=25 (too large) for comparison
fig_heat_r25 = px.density_map(
    mdi_land,
    lat='latitud', lon='longitud',
    radius=25,
    center={'lat': -1.5, 'lon': -78.5}, zoom=5.4,
    map_style='open-street-map',
)
fig_heat_r25.update_layout(
    title='Density Heatmap of Detentions/Apprehensions — radius=25<br>'
          '<sub>Compare to radius=10 above — note how much less local detail survives</sub>',
    margin=dict(l=0, r=0, t=60, b=0), height=650,
)
fig_heat_r25.show()


### Answer

At **radius=10**, the heatmap resolves distinct hotspots: Quito and Guayaquil stand out as the two brightest, most saturated cores (5,723 and 5,453 incidents respectively — by far the two largest cantons in the data), with visibly separate, smaller warm spots around Santo Domingo, Ibarra, Cuenca, and other provincial capitals.

At **radius=25**, each point's contribution spreads over a much wider area, so nearby hotspots start to bleed into each other — the corridor between Guayaquil, Santo Domingo, and Quito begins to merge into one smeared warm region, and the distinction between a genuinely dense city center and a moderately busy province gets lost. A radius that's too large doesn't add information, it destroys it: it trades true local resolution for a smoother-looking but less accurate picture, the opposite of what a density estimate is supposed to buy you.

Compared to the other two idioms: Section 11's dot density map already showed Quito and Guayaquil as the densest point clusters, but at ~38k overlapping points it's impossible to tell *how much* denser Quito is than, say, Cuenca — above a certain density everything just looks like a solid blob. The heatmap's continuous color scale fixes exactly that, showing a genuine intensity gradient rather than a saturated mass of overlapping dots. Section 9's choropleth told a different story altogether (conservation hectares, not incidents) and was constrained to canton *polygons* — it could never show concentration *within* a canton the way a continuous surface can. So the density heatmap is the one idiom of the three that shows both a comparable intensity value and within-region detail unconstrained by administrative boundaries — though it still carries the same caveat Section 12 raised: it's raw incident density, not a population-adjusted rate, so part of what makes Quito/Guayaquil "hot" here is still simply that a lot of people live there.
